# Baseline capability ladder on bundled ASAS-SN light curves

This notebook compares the baseline implementations in the order they were introduced:

1. global median;
2. per-camera rolling median;
3. unmasked per-camera GP;
4. the previous masked GP (camera-local masking plus late-onset consensus); and
5. the current masked GP, which additionally propagates an excursion interval when at least two same-band cameras independently corroborate it.

The first comparison uses real bundled light curves. A stage-by-stage view then separates the stiff GP, the previous camera-local mask, newly propagated points, late-onset consensus, the final quiescent baseline, and standardized residuals. The final synthetic example demonstrates the explicitly opt-in, colour-calibrated cross-band consensus path.

For the controlled old-versus-new comparison, the notebook temporarily disables only the current code's corroborated-interval combiner. Every other implementation detail and default remains identical.

In [ ]:
from contextlib import contextmanager
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

candidate_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
repo_root = next(
    (
        path for path in candidate_roots
        if (path / 'pyproject.toml').is_file()
        and (path / 'malca' / 'core' / 'baseline.py').is_file()
    ),
    None,
)
if repo_root is None:
    raise RuntimeError(f'Could not find the MALCA repository above {Path.cwd()}')

for path in (repo_root, repo_root / 'malca'):
    resolved = str(path.resolve())
    if resolved not in sys.path:
        sys.path.insert(0, resolved)

import malca.core.baseline as baseline_module
from malca.core.baseline import (
    global_median_baseline,
    per_camera_gp_baseline,
    per_camera_gp_baseline_masked,
    per_camera_median_baseline,
)
from malca.core.utils import clean_lc
from malca.io.lightcurve_io import (
    load_lightcurve_df,
    stable_camera_color,
    to_asassn_algorithm_frame,
)
from malca.stv.events import DEFAULT_BASELINE_KWARGS

JD_OFFSET = 2458000.0


In [ ]:
run_roots = [
    repo_root / 'output' / 'runs' / 'dat3-full-extended_2026-07-01-v4',
    repo_root / 'output' / 'runs' / 'runs_march18_bundle_all',
]
lightcurve_dirs = [root / 'bundle_assets' / 'lightcurves' for root in run_roots]
lightcurve_dirs = [path for path in lightcurve_dirs if path.is_dir()]
if not lightcurve_dirs:
    raise FileNotFoundError('No bundled light-curve directory was found under output/runs.')

target_ids = [
    '197569146752',  # strong corroborated-propagation stress case
    '214748665650',  # propagation plus same-band late-onset consensus
    '489626721133',
    '635656111241',
    '438086746412',
]
target_paths = {}
for target_id in target_ids:
    matches = [path / f'{target_id}.dat3' for path in lightcurve_dirs]
    target_paths[target_id] = next((path for path in matches if path.is_file()), None)

missing = [target_id for target_id, path in target_paths.items() if path is None]
if missing:
    raise FileNotFoundError(f'Missing requested bundled light curves: {missing}')

display(pd.DataFrame({
    'asas_sn_id': target_ids,
    'lightcurve_path': [str(target_paths[target_id]) for target_id in target_ids],
}))


In [ ]:
@contextmanager
def previous_masking_behavior():
    """Reconstruct the immediately previous masked-GP behavior."""
    original = baseline_module._corroborated_intervals

    def no_corroborated_intervals(*args, **kwargs):
        return []

    baseline_module._corroborated_intervals = no_corroborated_intervals
    try:
        yield
    finally:
        baseline_module._corroborated_intervals = original


def load_prepared_lightcurve(path):
    canonical = load_lightcurve_df(path, apply_quality=True)
    if canonical is None or canonical.empty:
        raise ValueError(f'No light-curve rows were loaded from {path}')
    algorithm_frame = to_asassn_algorithm_frame(canonical)
    return clean_lc(algorithm_frame).sort_values('JD').reset_index(drop=True)


def run_capability_ladder(lightcurve):
    results = {
        '1. Global median': global_median_baseline(lightcurve).reset_index(drop=True),
        '2. Per-camera rolling median': per_camera_median_baseline(lightcurve).reset_index(drop=True),
        '3. Unmasked per-camera GP': per_camera_gp_baseline(
            lightcurve, **DEFAULT_BASELINE_KWARGS
        ).reset_index(drop=True),
    }
    with previous_masking_behavior():
        results['4. Previous masked GP'] = per_camera_gp_baseline_masked(
            lightcurve, **DEFAULT_BASELINE_KWARGS
        ).reset_index(drop=True)
    results['5. Current masked GP + propagation'] = per_camera_gp_baseline_masked(
        lightcurve, **DEFAULT_BASELINE_KWARGS
    ).reset_index(drop=True)
    return results


def robust_scatter(values):
    values = pd.to_numeric(pd.Series(values), errors='coerce').to_numpy(float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.nan
    center = float(np.median(values))
    return float(1.4826 * np.median(np.abs(values - center)))


def summarize_ladder(target_id, lightcurve, results):
    previous = results['4. Previous masked GP']
    current = results['5. Current masked GP + propagation']
    previous_mask = previous['is_masked'].fillna(False).to_numpy(bool)
    current_mask = current['is_masked'].fillna(False).to_numpy(bool)
    rows = []
    for method, result in results.items():
        masked_points = (
            int(result['is_masked'].fillna(False).sum())
            if 'is_masked' in result.columns else 0
        )
        consensus_cameras = (
            int(result.groupby('camera#')['needs_consensus'].any().sum())
            if 'needs_consensus' in result.columns else 0
        )
        rows.append({
            'asas_sn_id': target_id,
            'points': len(lightcurve),
            'cameras': lightcurve['camera#'].nunique(),
            'method': method,
            'residual_MAD_mag': robust_scatter(result['resid']),
            'masked_points': masked_points,
            'newly_propagated_points': (
                int(np.sum(current_mask & ~previous_mask))
                if method.startswith('5.') else 0
            ),
            'consensus_cameras': consensus_cameras,
            'baseline_sources': ', '.join(
                sorted(result['baseline_source'].dropna().astype(str).unique())
            ),
        })
    return pd.DataFrame(rows)


def plot_camera_points(ax, frame, y_column, *, alpha=0.35):
    for camera, sub in frame.groupby('camera#', sort=True):
        sub = sub.sort_values('JD')
        ax.errorbar(
            sub['JD'] - JD_OFFSET, sub[y_column],
            yerr=sub['error'] if y_column == 'mag' else None,
            fmt='.', ms=3.5, lw=0.4, alpha=alpha,
            color=stable_camera_color(camera),
        )


def overlay_mask(ax, frame, mask, y_column, *, color, marker, label):
    mask = np.asarray(mask, bool)
    if not mask.any():
        return
    ax.scatter(
        frame.loc[mask, 'JD'] - JD_OFFSET, frame.loc[mask, y_column],
        s=24, marker=marker, color=color, linewidths=0.9, zorder=5, label=label,
    )


def make_ladder_figure(target_id, lightcurve, results):
    fig, axes = plt.subplots(
        len(results), 2, figsize=(18, 19), sharex='col', sharey='col'
    )
    previous = results['4. Previous masked GP']
    current = results['5. Current masked GP + propagation']
    previous_mask = previous['is_masked'].fillna(False).to_numpy(bool)
    current_mask = current['is_masked'].fillna(False).to_numpy(bool)
    retained_local = current_mask & previous_mask
    propagation_only = current_mask & ~previous_mask

    for row, (method, result) in enumerate(results.items()):
        magnitude_ax, residual_ax = axes[row]
        plot_camera_points(magnitude_ax, result, 'mag')
        plot_camera_points(residual_ax, result, 'resid')
        for camera, sub in result.groupby('camera#', sort=True):
            sub = sub.sort_values('JD')
            magnitude_ax.plot(
                sub['JD'] - JD_OFFSET, sub['baseline'],
                color=stable_camera_color(camera), lw=1.4,
            )

        if method.startswith('4.'):
            overlay_mask(
                magnitude_ax, result, previous_mask, 'mag',
                color='#d73027', marker='x', label='camera-local mask',
            )
            overlay_mask(
                residual_ax, result, previous_mask, 'resid',
                color='#d73027', marker='x', label='camera-local mask',
            )
        elif method.startswith('5.'):
            for ax, column in ((magnitude_ax, 'mag'), (residual_ax, 'resid')):
                overlay_mask(
                    ax, result, retained_local, column,
                    color='#d73027', marker='x', label='retained local mask',
                )
                overlay_mask(
                    ax, result, propagation_only, column,
                    color='#7b3294', marker='+', label='newly propagated',
                )

        residual_ax.axhline(0.0, color='0.25', lw=0.8)
        magnitude_ax.set_title(method, loc='left')
        residual_ax.set_title(f'{method} residual', loc='left')
        magnitude_ax.set_ylabel('magnitude')
        residual_ax.set_ylabel('residual [mag]')
        magnitude_ax.tick_params(direction='in', top=True, right=True)
        residual_ax.tick_params(direction='in', top=True, right=True)

    axes[0, 0].invert_yaxis()
    axes[0, 1].invert_yaxis()
    axes[-1, 0].set_xlabel('JD - 2458000')
    axes[-1, 1].set_xlabel('JD - 2458000')
    axes[-1, 0].legend(loc='best')
    axes[-1, 1].legend(loc='best')
    fig.suptitle(f'Baseline capability ladder: ASAS-SN {target_id}', fontsize=16)
    fig.tight_layout(rect=(0, 0, 1, 0.98))
    return fig


## Successive comparison on real bundled light curves

The same cleaned observations and the same live defaults are supplied to every method. Red crosses show the previous/current camera-local mask where it remains active. Purple plus signs show points excluded only because the newest same-band corroboration step propagated an interval into that camera. The residual MAD is descriptive only: a flexible baseline can lower it by absorbing real variability, so a smaller value is not automatically a better baseline.

In [ ]:
comparison_runs = {}
summary_tables = []

for target_id in target_ids:
    lightcurve = load_prepared_lightcurve(target_paths[target_id])
    results = run_capability_ladder(lightcurve)
    comparison_runs[target_id] = {'lightcurve': lightcurve, 'results': results}
    summary_tables.append(summarize_ladder(target_id, lightcurve, results))

    figure = make_ladder_figure(target_id, lightcurve, results)
    display(figure)
    plt.close(figure)

summary = pd.concat(summary_tables, ignore_index=True)
display(summary)


## Current masked-GP stages

These panels retain the notebook's original stage-by-stage presentation while separating the new propagation step from the older camera-local masking. `197569146752` is the propagation stress case; `214748665650` also shows late-onset same-band consensus.

In [ ]:
def plot_stage_data(ax, frame, y_column):
    plot_camera_points(ax, frame, y_column, alpha=0.32)
    ax.tick_params(direction='in', top=True, right=True)


def plot_camera_curve(ax, frame, column, *, linestyle='-', linewidth=1.4):
    for camera, sub in frame.groupby('camera#', sort=True):
        sub = sub.sort_values('JD')
        values = pd.to_numeric(sub[column], errors='coerce').to_numpy(float)
        if np.isfinite(values).any():
            ax.plot(
                sub['JD'] - JD_OFFSET, values,
                linestyle=linestyle, lw=linewidth, color=stable_camera_color(camera),
            )


def make_stage_figure(target_id, lightcurve, previous, current):
    fig, axes = plt.subplots(2, 3, figsize=(20, 11), sharex=True)
    ax_rough, ax_local, ax_propagated, ax_consensus, ax_final, ax_sigma = axes.flat
    previous_mask = previous['is_masked'].fillna(False).to_numpy(bool)
    current_mask = current['is_masked'].fillna(False).to_numpy(bool)
    retained_local = current_mask & previous_mask
    propagation_only = current_mask & ~previous_mask

    plot_stage_data(ax_rough, current, 'mag')
    plot_camera_curve(ax_rough, current, 'base_rough', linestyle=':')
    ax_rough.set_title('1. Stiff GP (base_rough)')

    plot_stage_data(ax_local, previous, 'mag')
    plot_camera_curve(ax_local, previous, 'baseline')
    overlay_mask(
        ax_local, previous, previous_mask, 'mag',
        color='#d73027', marker='x', label='previous camera-local mask',
    )
    ax_local.set_title('2. Previous camera-local masking')

    plot_stage_data(ax_propagated, current, 'mag')
    plot_camera_curve(ax_propagated, current, 'baseline')
    overlay_mask(
        ax_propagated, current, retained_local, 'mag',
        color='#d73027', marker='x', label='retained local mask',
    )
    overlay_mask(
        ax_propagated, current, propagation_only, 'mag',
        color='#7b3294', marker='+', label='newly propagated',
    )
    ax_propagated.set_title('3. Corroborated same-band propagation')

    plot_stage_data(ax_consensus, current, 'mag')
    plot_camera_curve(ax_consensus, current, 'base_consensus', linestyle='--', linewidth=1.8)
    consensus_cameras = int(current.groupby('camera#')['needs_consensus'].any().sum())
    if consensus_cameras == 0:
        ax_consensus.text(
            0.5, 0.08, 'No camera required consensus',
            transform=ax_consensus.transAxes, ha='center', va='bottom',
        )
    ax_consensus.set_title(f'4. Late-onset consensus ({consensus_cameras} cameras)')

    plot_stage_data(ax_final, current, 'mag')
    plot_camera_curve(ax_final, current, 'baseline')
    ax_final.set_title('5. Final quiescent baseline')

    plot_stage_data(ax_sigma, current, 'sigma_resid')
    ax_sigma.axhline(0, color='0.25', lw=0.8)
    ax_sigma.axhline(3, color='0.5', lw=0.8, ls='--')
    ax_sigma.axhline(-3, color='0.5', lw=0.8, ls='--')
    overlay_mask(
        ax_sigma, current, propagation_only, 'sigma_resid',
        color='#7b3294', marker='+', label='newly propagated',
    )
    ax_sigma.set_title('6. Standardized residual using sigma_eff')

    for ax in (ax_rough, ax_local, ax_propagated, ax_consensus, ax_final):
        ax.invert_yaxis()
        ax.set_ylabel('magnitude')
    ax_sigma.invert_yaxis()
    ax_sigma.set_ylabel('standardized residual')
    for ax in axes[-1]:
        ax.set_xlabel('JD - 2458000')
    ax_local.legend(loc='best')
    ax_propagated.legend(loc='best')
    ax_sigma.legend(loc='best')
    fig.suptitle(f'Current masked-GP stages: ASAS-SN {target_id}', fontsize=16)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    return fig


for target_id in ('197569146752', '214748665650'):
    run = comparison_runs[target_id]
    previous = run['results']['4. Previous masked GP']
    current = run['results']['5. Current masked GP + propagation']
    figure = make_stage_figure(target_id, run['lightcurve'], previous, current)
    display(figure)
    plt.close(figure)


## Opt-in calibrated cross-band consensus

Same-band consensus is automatic for qualifying late-onset cameras. Cross-band transfer remains disabled by default and must be requested explicitly. When enabled, it requires sufficient temporal overlap, estimates the inter-band magnitude offset robustly, and records the offset, scatter, overlap count, and calibration flag rather than copying a baseline between filters without calibration. The deterministic example below makes the expected one-magnitude offset visible.

In [ ]:
def make_cross_band_example(seed=711):
    rng = np.random.default_rng(seed)
    rows = []
    dip_center = 9600.0
    for camera in ('g1', 'g2'):
        jd = np.arange(7000.0, 11000.0, 5.0)
        mag = 14.0 + rng.normal(0.0, 0.015, len(jd))
        mag += 0.45 * np.exp(-0.5 * ((jd - dip_center) / 15.0) ** 2)
        rows.extend(
            {'JD': t, 'mag': m, 'error': 0.015, 'camera#': camera,
             'v_g_band': 0, 'saturated': 0}
            for t, m in zip(jd, mag)
        )
    for camera in ('v1', 'v2'):
        jd = np.arange(9500.0, 11000.0, 5.0)
        mag = 15.0 + rng.normal(0.0, 0.015, len(jd))
        mag += 0.45 * np.exp(-0.5 * ((jd - dip_center) / 15.0) ** 2)
        rows.extend(
            {'JD': t, 'mag': m, 'error': 0.015, 'camera#': camera,
             'v_g_band': 1, 'saturated': 0}
            for t, m in zip(jd, mag)
        )
    return pd.DataFrame(rows).sort_values('JD').reset_index(drop=True)


cross_band_input = make_cross_band_example()
cross_band_input['JD'] += JD_OFFSET
without_transfer = per_camera_gp_baseline_masked(
    cross_band_input,
    late_onset_buffer_days=300.0,
    min_anchor_overlap_days=30.0,
    allow_cross_band_consensus=False,
).reset_index(drop=True)
with_transfer = per_camera_gp_baseline_masked(
    cross_band_input,
    late_onset_buffer_days=300.0,
    min_anchor_overlap_days=30.0,
    allow_cross_band_consensus=True,
    cross_band_min_overlap_points=50,
).reset_index(drop=True)

v_band = with_transfer['camera#'].astype(str).str.startswith('v')
calibration_summary = (
    with_transfer.loc[v_band, [
        'camera#', 'baseline_source', 'cross_band_calibrated',
        'cross_band_offset_mag', 'cross_band_offset_scatter',
        'cross_band_overlap_points',
    ]]
    .drop_duplicates()
    .sort_values('camera#')
)
display(calibration_summary)

assert with_transfer.loc[v_band, 'cross_band_calibrated'].all()
assert np.isclose(
    np.nanmedian(with_transfer.loc[v_band, 'cross_band_offset_mag']), 1.0, atol=0.05
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharex=True, sharey=True)
for ax, result, title in (
    (axes[0], without_transfer, 'Cross-band transfer disabled (default)'),
    (axes[1], with_transfer, 'Calibrated cross-band consensus enabled'),
):
    selected = result['camera#'].astype(str).str.startswith('v')
    frame = result.loc[selected]
    plot_camera_points(ax, frame, 'mag', alpha=0.35)
    plot_camera_curve(ax, frame, 'baseline')
    ax.set_title(title)
    ax.set_xlabel('synthetic JD - 2458000')
    ax.set_ylabel('V-band magnitude')
    ax.tick_params(direction='in', top=True, right=True)
axes[0].invert_yaxis()
g_band = with_transfer['camera#'].astype(str).str.startswith('g')
for camera, sub in with_transfer.loc[g_band].groupby('camera#', sort=True):
    sub = sub.sort_values('JD')
    axes[1].plot(
        sub['JD'] - JD_OFFSET, sub['baseline'],
        color='0.35', lw=1.0, ls=':',
        label='unshifted g-band anchor' if camera == 'g1' else None,
    )
offset = float(np.nanmedian(with_transfer.loc[v_band, 'cross_band_offset_mag']))
axes[1].text(
    0.98, 0.06, f'calibrated offset = {offset:+.3f} mag',
    transform=axes[1].transAxes, ha='right', va='bottom',
)
axes[1].legend(loc='center right')
fig.tight_layout()
display(fig)
plt.close(fig)
